# Assign Unmatched RL Roles to SASB Categories

**Two-step approach (at `role_k10000_v3` level):**
1. **Step 1 (LLM role-based):** `gemma_role_classification_output.csv` — gemma classified each unique role triple by title. Group by `role_k10000_v3`, union all assigned SASB categories.
2. **Step 2 (description-based fallback):** For `role_k10000_v3` labels that step 1 never mapped, use `gemma_combined_classification_output_100k.csv` (10 jobs per `role_k10000_v3`). Assign a SASB category if hit rate ≥ `MIN_RATE` (e.g. ≥3 out of 10 descriptions tagged).

**Output:** `role_to_sasb_mapping.csv` — one row per `role_k10000_v3`.

In [2]:
import ast
import pandas as pd
from pathlib import Path

BASE      = Path('D:/Dropbox/fengheliu/temp/sasb_jobs')
TEMP_DATA = BASE / 'temp_data'

ROLE_LLM_OUT = TEMP_DATA / 'step1_gemma_role_classification' / 'gemma_role_classification_output.csv'
DESC_LLM_OUT = TEMP_DATA / 'step2_gemma_combined_classification' / 'gemma_combined_classification_output_10m_llama_cpp.csv'
SAMPLE       = TEMP_DATA / 'step2_gemma_combined_classification' / 'role_stratified_sample.csv'

ROLE_COL = 'role_k10000_v3'

# With 10 jobs per role, MIN_RATE=0.20 means >=2/10 descriptions tagged
# MIN_RATE=0.30 means >=3/10 — more conservative
MIN_N    = 10     # minimum jobs with that role in the 100k sample (handles edge cases)
MIN_RATE = 0.5  # fraction of descriptions that must be tagged

OUTPUT = TEMP_DATA / 'step3_assign_unmatched_roles' / f'role_to_sasb_mapping_10m_{MIN_RATE}.csv'

def parse_list(val):
    if isinstance(val, list): return val
    if isinstance(val, str) and val.strip():
        try: return ast.literal_eval(val)
        except: return []
    return []

## EDA — gemma_combined_classification_output_100k.csv

In [3]:
eda = pd.read_csv(DESC_LLM_OUT)
eda['sasb_categories'] = eda['sasb_categories'].apply(parse_list)
eda['n_cats']          = eda['sasb_categories'].apply(len)

total = len(eda)
n_pos = eda['n_cats'].gt(0).sum()
print(f"Total rows            : {total:,}")
print(f"Overall hit rate      : {n_pos:,} / {total:,} = {n_pos/total*100:.2f}%")
print(f"Avg categories (pos)  : {eda.loc[eda['n_cats']>0,'n_cats'].mean():.2f}")
print()

# Distribution of n_cats per job
print("Distribution of #categories assigned per job:")
print(eda['n_cats'].value_counts().sort_index().rename('n_jobs').to_frame().to_string())
print()

# Hit rate per SASB category
from collections import Counter
cat_counts = Counter(cat for cats in eda['sasb_categories'] for cat in cats)
cat_df = (
    pd.DataFrame.from_dict(cat_counts, orient='index', columns=['n_jobs'])
    .sort_values('n_jobs', ascending=False)
    .assign(hit_rate_pct=lambda d: (d['n_jobs'] / total * 100).round(3))
)
cat_df.index.name = 'sasb_category'
print(f"Hit rate by SASB category (out of {total:,} total jobs):")
display(cat_df)
print()

# Distribution of number of jobs per role (should be ~10 each)
sample_eda = pd.read_csv(SAMPLE, usecols=['position_id', ROLE_COL])
jobs_per_role = (
    sample_eda.merge(eda[['position_id']], on='position_id', how='inner')
    [ROLE_COL].value_counts()
)
print(f"Jobs per {ROLE_COL} — summary (expected ~10 each):")
print(jobs_per_role.describe().to_string())
print()
print("Distribution of jobs-per-role counts:")
print(jobs_per_role.value_counts().sort_index().rename('n_roles').to_frame().to_string())


Total rows            : 7,349,879
Overall hit rate      : 1,935,574 / 7,349,879 = 26.33%
Avg categories (pos)  : 1.19

Distribution of #categories assigned per job:
         n_jobs
n_cats         
0       5414305
1       1607038
2        302359
3         19046
4          4331
5          1862
6           695
7           185
8            49
9             8
10            1

Hit rate by SASB category (out of 7,349,879 total jobs):


,n_jobs,hit_rate_pct
sasb_category,,
Employee_Health_&_Safety,310492,4.224
Product_Quality_&_Safety,278415,3.788
Business_Ethics,195917,2.666
Data_Security,173595,2.362
Energy_Management,157861,2.148
Supply_Chain_Management,124199,1.690
Management_of_the_Legal_&_Regulatory_Environment,117169,1.594
Waste_&_Hazardous_Materials_Management,105530,1.436
Human_Rights_&_Community_Relations,92508,1.259



Jobs per role_k10000_v3 — summary (expected ~10 each):
count    10000.000000
mean       734.987900
std        357.950816
min          1.000000
25%        398.000000
50%       1000.000000
75%       1000.000000
max       1000.000000

Distribution of jobs-per-role counts:
       n_roles
count         
1            6
2            1
3            2
4            3
5            5
6            4
7            3
8            4
9            5
10           3
11           8
12           5
13           3
14           9
15          11
16           7
17          10
18           5
19           9
20           7
21          12
22           9
23          13
24           5
25           4
26           8
27          11
28           5
29          11
30          11
31           8
32           9
33           7
34          11
35          10
36           6
37          14
38           9
39          12
40           7
41           4
42          10
43           8
44          11
45           7
46          18
47       

## Step 1 — Role-title LLM mapping
Group `gemma_role_classification_output.csv` by `role_k10000_v3`, union all assigned SASB categories.

In [4]:
role_df = pd.read_csv(ROLE_LLM_OUT)
role_df['sasb_categories'] = role_df['sasb_categories'].apply(parse_list)
role_df = role_df[role_df[ROLE_COL].notna()]

print(f'Loaded {len(role_df):,} rows, {role_df[ROLE_COL].nunique():,} unique {ROLE_COL}')

step1 = (
    role_df.groupby(ROLE_COL)['sasb_categories']
    .agg(lambda x: sorted({cat for cats in x for cat in cats}))
    .reset_index()
)
step1['source'] = step1['sasb_categories'].apply(lambda x: 'gemma_role' if x else 'unmatched')

matched   = step1[step1['source'] == 'gemma_role']
unmatched = step1[step1['source'] == 'unmatched']

print(f'Step 1 — matched   : {len(matched):,} role labels')
print(f'Step 1 — unmatched : {len(unmatched):,} role labels')
matched.head(10)

Loaded 10,002 rows, 10,002 unique role_k10000_v3
Step 1 — matched   : 2,411 role labels
Step 1 — unmatched : 7,591 role labels


,role_k10000_v3,sasb_categories,source
3,"""IoT Product Manager""",[Product_Design_&_Lifecycle_Management],gemma_role
7,3D Artist,[Product_Design_&_Lifecycle_Management],gemma_role
8,3D Designer,[Product_Design_&_Lifecycle_Management],gemma_role
9,3D Printing Engineer,[Materials_Sourcing_&_Efficiency],gemma_role
30,AML Analyst,[Business_Ethics],gemma_role
53,Academic Coach,[Employee_Health_&_Safety],gemma_role
63,Access Administrator,[Data_Security],gemma_role
64,Access Analyst,[Access_&_Affordability],gemma_role
65,Access Control Analyst,[Data_Security],gemma_role
67,Access Control Technician,[Data_Security],gemma_role


## Step 2 — Description-based fallback (100k sample)
Join `role_stratified_sample.csv` with `gemma_combined_classification_output_100k.csv` on `position_id`.
For each unmatched `role_k10000_v3`, compute per-SASB hit rate and assign if ≥ `MIN_RATE`.

In [5]:
sample   = pd.read_csv(SAMPLE, usecols=['position_id', ROLE_COL])
desc_llm = pd.read_csv(DESC_LLM_OUT, usecols=['position_id', 'sasb_categories'])
desc_llm['sasb_categories'] = desc_llm['sasb_categories'].apply(parse_list)

merged = sample.merge(desc_llm, on='position_id', how='inner')
print(f'Joined: {len(merged):,} rows, {merged[ROLE_COL].nunique():,} unique {ROLE_COL}')

unmatched_roles = set(unmatched[ROLE_COL])
df_unmatched    = merged[merged[ROLE_COL].isin(unmatched_roles)].copy()
print(f'Unmatched roles in sample : {df_unmatched[ROLE_COL].nunique():,} labels, {len(df_unmatched):,} jobs')

all_cats = sorted({cat for cats in df_unmatched['sasb_categories'] for cat in cats})
print(f'SASB categories seen      : {len(all_cats)}')

Joined: 7,349,879 rows, 10,000 unique role_k10000_v3
Unmatched roles in sample : 7,589 labels, 5,757,421 jobs
SASB categories seen      : 26


In [6]:
records = []
for role, grp in df_unmatched.groupby(ROLE_COL, dropna=False):
    n = len(grp)
    cat_rates = {
        cat: grp['sasb_categories'].apply(lambda x: cat in x).sum() / n
        for cat in all_cats
    }
    top_cat  = max(cat_rates, key=cat_rates.get) if cat_rates else None
    top_rate = cat_rates[top_cat] if top_cat else 0.0

    if n >= MIN_N:
        assigned = sorted(cat for cat, rate in cat_rates.items() if rate >= MIN_RATE)
        source   = 'gemma_desc' if assigned else 'below_threshold'
    else:
        assigned = []
        source   = 'too_few_samples'

    records.append({
        ROLE_COL:          role,
        'n_jobs':          n,
        'top_cat':         top_cat,
        'top_rate':        round(top_rate, 4),
        'sasb_categories': assigned,
        'source':          source,
    })

step2 = pd.DataFrame(records)
print(f'Step 2 — desc-assigned   : {(step2["source"] == "gemma_desc").sum()}')
print(f'Step 2 — too few samples : {(step2["source"] == "too_few_samples").sum()}')
print(f'Step 2 — below threshold : {(step2["source"] == "below_threshold").sum()}')
step2[step2['source'] == 'gemma_desc'].sort_values('top_rate', ascending=False).head(20)

Step 2 — desc-assigned   : 97
Step 2 — too few samples : 20
Step 2 — below threshold : 7472


,role_k10000_v3,n_jobs,top_cat,top_rate,sasb_categories,source
2684,Financial Crime Analyst,1000,Business_Ethics,0.9870,[Business_Ethics],gemma_desc
2882,Fraud Prevention,1000,Business_Ethics,0.9770,[Business_Ethics],gemma_desc
3797,Labor Attorney,545,Labor_Practices,0.9761,[Labor_Practices],gemma_desc
7402,Vulnerability Management Specialist,1000,Data_Security,0.9580,[Data_Security],gemma_desc
1751,Cybersecurity Sales,1000,Data_Security,0.9300,[Data_Security],gemma_desc
4721,Noise Consultant,13,Air_Quality,0.9231,[Air_Quality],gemma_desc
1804,Data Protection Engineer,1000,Data_Security,0.9200,[Data_Security],gemma_desc
1680,Customer Due Diligence Analyst,1000,Business_Ethics,0.9160,[Business_Ethics],gemma_desc
2084,Diversity Coordinator,1000,"Employee_Engagement,_Diversity_&_Inclusion",0.9110,"[Employee_Engagement,_Diversity_&_Inclusion]",gemma_desc
3401,Housing and Community Volunteer,39,Human_Rights_&_Community_Relations,0.8974,[Human_Rights_&_Community_Relations],gemma_desc


In [7]:
step2[step2['source'] == 'gemma_desc'].sort_values('top_rate', ascending=False).tail(25)

,role_k10000_v3,n_jobs,top_cat,top_rate,sasb_categories,source
6753,Talent Development Consultant,1000,"Employee_Engagement,_Diversity_&_Inclusion",0.5620,"[Employee_Engagement,_Diversity_&_Inclusion]",gemma_desc
7221,Utility Coordination Engineer,794,Energy_Management,0.5579,[Energy_Management],gemma_desc
2827,Food Technologist,1000,Product_Quality_&_Safety,0.5550,[Product_Quality_&_Safety],gemma_desc
3745,Jewelry Inspector,76,Product_Quality_&_Safety,0.5526,[Product_Quality_&_Safety],gemma_desc
3175,HR Business Partnering,1000,"Employee_Engagement,_Diversity_&_Inclusion",0.5520,"[Employee_Engagement,_Diversity_&_Inclusion, L...",gemma_desc
1756,Cytogenetic Technician,456,Product_Quality_&_Safety,0.5482,[Product_Quality_&_Safety],gemma_desc
3421,IT Asset Manager,1000,Data_Security,0.5460,[Data_Security],gemma_desc
5117,Personal Injury Attorney,335,Product_Quality_&_Safety,0.5373,[Product_Quality_&_Safety],gemma_desc
257,Analytical Chemistry Scientist,1000,Product_Quality_&_Safety,0.5350,[Product_Quality_&_Safety],gemma_desc
2347,Energy Sales Representative,1000,Energy_Management,0.5350,[Energy_Management],gemma_desc


## Inspect: roles near but below the threshold

In [8]:
near = step2[
    (step2['source'] == 'below_threshold') &
    (step2['top_rate'] >= 0.40)
].sort_values('top_rate', ascending=False)

print(f'Roles below threshold but top_rate >= 40%: {len(near)}')
near[['role_k10000_v3', 'n_jobs', 'top_cat', 'top_rate']].tail(15)

Roles below threshold but top_rate >= 40%: 99


,role_k10000_v3,n_jobs,top_cat,top_rate
807,Business Control,1000,Business_Ethics,0.4140
871,CSR Professional,1000,Human_Rights_&_Community_Relations,0.4120
1617,Cross Dock Coordinator,1000,Employee_Health_&_Safety,0.4090
3491,In Vivo Pharmacologist,250,Product_Quality_&_Safety,0.4080
3185,HR Operation,1000,Labor_Practices,0.4080
4164,Maintenance Lead,1000,Energy_Management,0.4070
4488,Metering Coordinator,362,Energy_Management,0.4061
2495,Exercise Physiologist,1000,Employee_Health_&_Safety,0.4060
3089,Good Clinical Practice Auditor,272,Product_Quality_&_Safety,0.4044
6782,Tax and Contract Lawyer,1000,Business_Ethics,0.4040


## Threshold sensitivity (how many roles get mapped at each cutoff)

In [10]:
eligible = step2[step2['n_jobs'] >= MIN_N]

rows = []
for thresh in [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]:
    n_desc = (eligible['top_rate'] >= thresh).sum()
    rows.append({
        'min_rate':     thresh,
        'desc_assigned': n_desc,
        'total_mapped': len(matched) + n_desc,
        'pct_of_unmatched': f"{n_desc/len(eligible)*100:.1f}%" if len(eligible) else 'N/A'
    })

pd.DataFrame(rows)

,min_rate,desc_assigned,total_mapped,pct_of_unmatched
0,0.1,2533,4944,33.5%
1,0.2,910,3321,12.0%
2,0.3,402,2813,5.3%
3,0.4,196,2607,2.6%
4,0.5,97,2508,1.3%
5,0.6,60,2471,0.8%
6,0.7,38,2449,0.5%
7,0.8,18,2429,0.2%


## Combine Step 1 + Step 2 → final mapping

In [11]:
role_counts = merged[ROLE_COL].value_counts().rename('n_jobs_100k')

step1_out = matched[[ROLE_COL, 'sasb_categories', 'source']].copy()
step1_out['n_jobs'] = step1_out[ROLE_COL].map(role_counts)

step2_out = step2[[ROLE_COL, 'n_jobs', 'sasb_categories', 'source']].copy()

mapping = pd.concat(
    [step1_out[[ROLE_COL, 'n_jobs', 'sasb_categories', 'source']],
     step2_out],
    ignore_index=True
).sort_values(ROLE_COL).reset_index(drop=True)

has_mapping = mapping['sasb_categories'].apply(len) > 0
print(f'Total {ROLE_COL} labels   : {len(mapping):,}')
print(f'  With SASB mapping     : {has_mapping.sum():,} ({has_mapping.mean()*100:.1f}%)')
print(f'    from gemma_role     : {(mapping["source"]=="gemma_role").sum():,}')
print(f'    from gemma_desc     : {(mapping["source"]=="gemma_desc").sum():,}')
print(f'  No mapping            : {(~has_mapping).sum():,} ({(~has_mapping).mean()*100:.1f}%)')

mapping[has_mapping].head(20)

Total role_k10000_v3 labels   : 10,000
  With SASB mapping     : 2,508 (25.1%)
    from gemma_role     : 2,411
    from gemma_desc     : 97
  No mapping            : 7,492 (74.9%)


,role_k10000_v3,n_jobs,sasb_categories,source
3,"""IoT Product Manager""",1000,[Product_Design_&_Lifecycle_Management],gemma_role
7,3D Artist,1000,[Product_Design_&_Lifecycle_Management],gemma_role
8,3D Designer,1000,[Product_Design_&_Lifecycle_Management],gemma_role
9,3D Printing Engineer,1000,[Materials_Sourcing_&_Efficiency],gemma_role
30,AML Analyst,1000,[Business_Ethics],gemma_role
53,Academic Coach,874,[Employee_Health_&_Safety],gemma_role
63,Access Administrator,1000,[Data_Security],gemma_role
64,Access Analyst,1000,[Access_&_Affordability],gemma_role
65,Access Control Analyst,694,[Data_Security],gemma_role
67,Access Control Technician,247,[Data_Security],gemma_role


## Save

In [12]:
mapping.to_csv(OUTPUT, index=False)
print(f'Saved {len(mapping):,} rows to: {OUTPUT}')
mapping[has_mapping]

Saved 10,000 rows to: D:\Dropbox\fengheliu\temp\sasb_jobs\temp_data\step3_assign_unmatched_roles\role_to_sasb_mapping_10m_0.5.csv


,role_k10000_v3,n_jobs,sasb_categories,source
3,"""IoT Product Manager""",1000,[Product_Design_&_Lifecycle_Management],gemma_role
7,3D Artist,1000,[Product_Design_&_Lifecycle_Management],gemma_role
8,3D Designer,1000,[Product_Design_&_Lifecycle_Management],gemma_role
9,3D Printing Engineer,1000,[Materials_Sourcing_&_Efficiency],gemma_role
30,AML Analyst,1000,[Business_Ethics],gemma_role
...,...,...,...,...
9982,Youth Transition Coordinator,1000,[Human_Rights_&_Community_Relations],gemma_role
9985,Zoning Analyst,126,[Ecological_Impacts],gemma_role
9986,Zoo Maintenance Technician,209,[Ecological_Impacts],gemma_role
9987,Zoo Supervisor,574,[Employee_Health_&_Safety],gemma_role
